# 07 - Local SHAP Analysis

**Dissertation:** Explainable Machine Learning for Locational Frequency Stability Assessment (IEEE 9-bus)

This notebook performs local (instance-level) SHAP analysis: identifying critical scenarios near grid-code thresholds, waterfall and dependence plots with unscaled feature values, cross-bus comparison, and narrative summaries (Kilembe, Hamilton & Papadopoulos 2025).

## Cell 1 - Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb

BASE_PATH = '/content/drive/MyDrive/Dissertation_ML_Project_V1'
os.chdir(BASE_PATH)

FEATURE_NAMES = ['SG 1 MVA', 'SG 2 MVA', 'SG 3 MVA', 'System Loading', 'CIG MW', 'Outage SG', 'SG 1 MW', 'SG 2 MW', 'SG 3 MW']
TARGET_ORDER = (
    [f'RoCoF Bus {i}' for i in range(1, 10)] + ['RoCoF Worst'] +
    [f'Nadir Bus {i}' for i in range(1, 10)] + ['Nadir Worst']
)

shap_dir = os.path.join(BASE_PATH, 'shap_results')
processed_dir = os.path.join(BASE_PATH, 'data', 'processed')

with open(os.path.join(shap_dir, 'all_shap_values.pkl'), 'rb') as f:
    all_shap = pickle.load(f)

if os.path.exists(os.path.join(shap_dir, 'X_test_raw.pkl')):
    X_test_raw = joblib.load(os.path.join(shap_dir, 'X_test_raw.pkl'))
else:
    X_test_raw = joblib.load(os.path.join(processed_dir, 'X_test_raw.pkl'))

X_test = joblib.load(os.path.join(processed_dir, 'X_test_scaled.pkl'))
X_test_values = X_test.values if hasattr(X_test, 'values') else np.asarray(X_test)

y_test_dict = joblib.load(os.path.join(processed_dir, 'y_test_dict.pkl'))
y_test = pd.DataFrame(y_test_dict)

rf_shap_path = os.path.join(shap_dir, 'rf', 'all_shap_values_rf.pkl')
xgb_shap_path = os.path.join(shap_dir, 'xgboost', 'all_shap_values_xgb.pkl')
if os.path.exists(rf_shap_path):
    with open(rf_shap_path, 'rb') as f:
        rf_shap = pickle.load(f)
else:
    rf_shap = {}

if os.path.exists(xgb_shap_path):
    with open(xgb_shap_path, 'rb') as f:
        xgb_shap = pickle.load(f)
else:
    xgb_shap = {}
ALL_ALG_SHAP = {'MLP': all_shap, 'RF': rf_shap, 'XGBoost': xgb_shap}

rocof_cols = [f'RoCoF Bus {i}' for i in range(1, 10)]
nadir_cols = [f'Nadir Bus {i}' for i in range(1, 10)]
if 'RoCoF Worst' not in y_test.columns and all(c in y_test.columns for c in rocof_cols):
    y_test['RoCoF Worst'] = y_test[rocof_cols].min(axis=1)
if 'Nadir Worst' not in y_test.columns and all(c in y_test.columns for c in nadir_cols):
    y_test['Nadir Worst'] = y_test[nadir_cols].min(axis=1)

n_test = len(X_test_raw)
if X_test_raw.index.dtype != np.int64 and not np.issubdtype(X_test_raw.index.dtype, np.integer):
    X_test_raw = X_test_raw.reset_index(drop=True)
if len(y_test) != n_test:
    raise AssertionError(
        f"y_test length mismatch: y_test has {len(y_test)} rows, "
        f"expected {n_test}. Stale or corrupted artefact. "
        f"Do not proceed with local analysis."
    )


def _fname_to_target(filename):
    return os.path.splitext(filename)[0].replace('_', ' ')


def _load_xgb_model(path_to_ubj, x_reference):
    model = xgb.XGBRegressor()
    model.load_model(path_to_ubj)
    assert callable(getattr(model, 'predict', None)), f'Loaded XGBoost model has no predict(): {path_to_ubj}'
    sample_pred = np.asarray(model.predict(x_reference[:1]))
    assert sample_pred.shape == (1,), (
        f'Loaded XGBoost model predict() failed for {path_to_ubj}: got shape {sample_pred.shape}.'
    )
    booster = model.get_booster()
    assert booster is not None and len(booster.get_dump()) > 0, (
        f'Loaded XGBoost model has no valid booster: {path_to_ubj}'
    )
    return model


models_dir = os.path.join(BASE_PATH, 'models', 'MLP')
models = {}
for filename in sorted(os.listdir(models_dir)):
    if filename.endswith('.pkl') and '_predictions' not in filename:
        models[_fname_to_target(filename)] = joblib.load(os.path.join(models_dir, filename))

rf_models_dir = os.path.join(BASE_PATH, 'models', 'RF')
rf_models = {}
if os.path.exists(rf_models_dir):
    for filename in sorted(os.listdir(rf_models_dir)):
        if filename.endswith('.pkl') and '_predictions' not in filename:
            rf_models[_fname_to_target(filename)] = joblib.load(os.path.join(rf_models_dir, filename))

xgb_models_dir = os.path.join(BASE_PATH, 'models', 'XGBoost')
xgb_models = {}
if os.path.exists(xgb_models_dir):
    for filename in sorted(os.listdir(xgb_models_dir)):
        if filename.endswith('.ubj'):
            target_name = _fname_to_target(filename)
            path = os.path.join(xgb_models_dir, filename)
            xgb_models[target_name] = _load_xgb_model(path, X_test_values)

fig_out = os.path.join(BASE_PATH, 'figures', 'SHAP_local')
os.makedirs(fig_out, exist_ok=True)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

print(
    f'Loaded MLP: {len(all_shap)} SHAP, {len(models)} models; '
    f'RF: {len(rf_shap)} SHAP, {len(rf_models)} models; '
    f'XGBoost: {len(xgb_shap)} SHAP, {len(xgb_models)} native .ubj models.'
)
print(f'X_test_raw: {X_test_raw.shape}, y_test: {y_test.shape}')
print(f'Output: {fig_out}')


## Cell 2 - Identify Critical Scenarios

In [ ]:

y_rocof_b2 = y_test['RoCoF Bus 2'].values
y_rocof_worst = y_test['RoCoF Worst'].values
y_nadir_b2 = y_test['Nadir Bus 2'].values
y_nadir_worst = y_test['Nadir Worst'].values

# Indices (iloc) for worst and threshold-near scenarios
idx_worst_rocof_b2 = np.argsort(y_rocof_b2)[:5]
idx_worst_rocof_sys = np.argsort(y_rocof_worst)[:5]
idx_worst_nadir_b2 = np.argsort(y_nadir_b2)[:5]
idx_worst_nadir_sys = np.argsort(y_nadir_worst)[:5]

# Near thresholds: RoCoF ~ -0.5 and -1.0
near_05 = np.argsort(np.abs(y_rocof_worst - (-0.5)))[:5]
near_10 = np.argsort(np.abs(y_rocof_worst - (-1.0)))[:5]


critical = {
    'worst_rocof_bus2': idx_worst_rocof_b2,
    'worst_rocof_sys': idx_worst_rocof_sys,
    'worst_nadir_bus2': idx_worst_nadir_b2,
    'worst_nadir_sys': idx_worst_nadir_sys,
    'near_rocof_05': near_05,
    'near_rocof_10': near_10,
    'near_nadir_limit': near_nadir,
}

i_worst_rocof_b2 = int(idx_worst_rocof_b2[0])
i_worst_rocof_sys = int(idx_worst_rocof_sys[0])
i_worst_nadir_b2 = int(idx_worst_nadir_b2[0])
i_worst_nadir_sys = int(idx_worst_nadir_sys[0])

median_rocof_b2 = np.median(y_rocof_b2)
typical_rocof_b2 = np.argmin(np.abs(y_rocof_b2 - median_rocof_b2))
median_nadir_b2 = np.median(y_nadir_b2)
typical_nadir_b2 = np.argmin(np.abs(y_nadir_b2 - median_nadir_b2))

# Single worst overall for cross-bus: use worst RoCoF system-wide scenario index
i_worst_overall_rocof = int(idx_worst_rocof_sys[0])
i_worst_overall_nadir = int(idx_worst_nadir_sys[0])

print('--- Worst 5 RoCoF at Bus 2 (most negative) ---')
for k in idx_worst_rocof_b2:
    print(f'  iloc {k}: RoCoF Bus 2 = {y_rocof_b2[k]:.4f} Hz/s')
print('\n--- Worst 5 system-wide RoCoF ---')
for k in idx_worst_rocof_sys:
    print(f'  iloc {k}: RoCoF Worst = {y_rocof_worst[k]:.4f} Hz/s')
print('\n--- Worst 5 Nadir at Bus 2 ---')
for k in idx_worst_nadir_b2:
    print(f'  iloc {k}: Nadir Bus 2 = {y_nadir_b2[k]:.4f} Hz')
print('\n--- Worst 5 system-wide Nadir ---')
for k in idx_worst_nadir_sys:
    print(f'  iloc {k}: Nadir Worst = {y_nadir_worst[k]:.4f} Hz')

print('\n--- Worst RoCoF Bus 2 scenario (iloc) - features (unscaled) and target ---')
row = X_test_raw.iloc[i_worst_rocof_b2]
print(row.to_string())
print(f'Actual RoCoF Bus 2: {y_rocof_b2[i_worst_rocof_b2]:.4f} Hz/s')

print('\n--- Worst Nadir Bus 2 scenario - features and target ---')
row = X_test_raw.iloc[i_worst_nadir_b2]
print(row.to_string())
print(f'Actual Nadir Bus 2: {y_nadir_b2[i_worst_nadir_b2]:.4f} Hz')

## Cell 3 - Waterfall Plots: Worst-Case RoCoF

In [ ]:
def make_explanation_one(all_shap, target_name, iloc_idx, X_test_raw, feature_names):
    ev = all_shap[target_name]['expected_value']
    sv = np.array(all_shap[target_name]['shap_values'])
    row = X_test_raw.iloc[iloc_idx]
    # Waterfall expects feature values as 1D array (length n_features), not (1, n_features)
    if hasattr(row, 'values'):
        data_1d = np.asarray(row.values).flatten()
    else:
        data_1d = np.asarray(row).flatten()
    return shap.Explanation(
        values=sv[iloc_idx],
        base_values=ev,
        data=data_1d,
        feature_names=feature_names
    )

# Worst RoCoF at Bus 2
ex = make_explanation_one(all_shap, 'RoCoF Bus 2', i_worst_rocof_b2, X_test_raw, FEATURE_NAMES)
shap.plots.waterfall(ex, show=False)
plt.title('MLP - SHAP Waterfall - Worst RoCoF Scenario at Bus 2')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_RoCoF_Bus2_worst_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

# Worst RoCoF system-wide
ex = make_explanation_one(all_shap, 'RoCoF Worst', i_worst_rocof_sys, X_test_raw, FEATURE_NAMES)
shap.plots.waterfall(ex, show=False)
plt.title('MLP - SHAP Waterfall - Worst System-Wide RoCoF Scenario')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_RoCoF_Worst_worst_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

# Typical (near-median) RoCoF at Bus 2
ex = make_explanation_one(all_shap, 'RoCoF Bus 2', typical_rocof_b2, X_test_raw, FEATURE_NAMES)
shap.plots.waterfall(ex, show=False)
plt.title('MLP - SHAP Waterfall - Typical RoCoF Scenario at Bus 2')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_RoCoF_Bus2_typical_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Analytical commentary ---')
print('Worst-case waterfalls show which features move the prediction toward the severe end of the test-set response. Typical scenarios show a more balanced contribution. Unscaled feature values allow physical interpretation in MVA and MW.')

## Cell 4 - Waterfall Plots: Worst-Case Nadir

In [ ]:
# Worst Nadir at Bus 2
ex = make_explanation_one(all_shap, 'Nadir Bus 2', i_worst_nadir_b2, X_test_raw, FEATURE_NAMES)
shap.plots.waterfall(ex, show=False)
plt.title('MLP - SHAP Waterfall - Worst Nadir Scenario at Bus 2')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_Nadir_Bus2_worst_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

# Worst Nadir system-wide
ex = make_explanation_one(all_shap, 'Nadir Worst', i_worst_nadir_sys, X_test_raw, FEATURE_NAMES)
shap.plots.waterfall(ex, show=False)
plt.title('MLP - SHAP Waterfall - Worst System-Wide Nadir Scenario')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_Nadir_Worst_worst_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

# Typical Nadir at Bus 2
ex = make_explanation_one(all_shap, 'Nadir Bus 2', typical_nadir_b2, X_test_raw, FEATURE_NAMES)
shap.plots.waterfall(ex, show=False)
plt.title('MLP - SHAP Waterfall - Typical Nadir Scenario at Bus 2')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_Nadir_Bus2_typical_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Analytical commentary ---')
print('Nadir waterfalls show which features lower the predicted minimum frequency. Comparing them with RoCoF highlights differences between initial frequency response and minimum-frequency behaviour.')

## Cell 5 - SHAP Dependence Plots (Top 3 Features at Bus 2)

In [ ]:
data_raw = X_test_raw[FEATURE_NAMES].values if isinstance(X_test_raw, pd.DataFrame) else X_test_raw

def top3_features(all_shap, target_name):
    sv = np.array(all_shap[target_name]['shap_values'])
    mean_abs = np.abs(sv).mean(axis=0)
    order = np.argsort(mean_abs)[::-1]
    return [FEATURE_NAMES[i] for i in order[:3]]

def make_full_explanation(all_shap, target_name, data_raw, feature_names):
    ev = all_shap[target_name]['expected_value']
    sv = np.array(all_shap[target_name]['shap_values'])
    return shap.Explanation(values=sv, base_values=np.full(sv.shape[0], ev),
                            data=data_raw, feature_names=feature_names)

def strongest_correlate(x_feat, data_raw, feature_names):
    """Non-self feature with the largest |Pearson r| against x_feat."""
    xi = feature_names.index(x_feat)
    x = data_raw[:, xi]
    best_feat, best_abs = None, -1.0
    for j, cand in enumerate(feature_names):
        if cand == x_feat:
            continue
        r = np.corrcoef(x, data_raw[:, j])[0, 1]
        if abs(r) > best_abs:
            best_abs, best_feat = abs(r), cand
    return best_feat

def dependence_panel(ax, x_feat, expl, data_raw, feature_names, color_feat):
    fi = feature_names.index(x_feat)
    cfi = feature_names.index(color_feat)
    sc = ax.scatter(data_raw[:, fi], expl.values[:, fi],
                    c=data_raw[:, cfi], cmap='coolwarm', alpha=0.6, s=15)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel(x_feat + ' (unscaled)')
    ax.set_ylabel('SHAP value')
    ax.set_title(x_feat)
    plt.colorbar(sc, ax=ax, label=color_feat)

#  RoCoF Bus 2: per-panel auto colour by strongest |r|
top3_rocof = top3_features(all_shap, 'RoCoF Bus 2')
expl_rocof = make_full_explanation(all_shap, 'RoCoF Bus 2', data_raw, FEATURE_NAMES)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for i, feat in enumerate(top3_rocof):
    color_feat = strongest_correlate(feat, data_raw, FEATURE_NAMES)
    dependence_panel(axes[i], feat, expl_rocof, data_raw, FEATURE_NAMES, color_feat)
plt.suptitle('MLP - SHAP dependence - RoCoF Bus 2 (top 3 features)')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'dependence_RoCoF_Bus2_MLP.png'),
            dpi=300, bbox_inches='tight')
plt.show()

#  Nadir Bus 2: all three panels coloured by SG 3 MW

COLOR_FEAT_NADIR = 'SG 3 MW'
top3_nadir = top3_features(all_shap, 'Nadir Bus 2')
expl_nadir = make_full_explanation(all_shap, 'Nadir Bus 2', data_raw, FEATURE_NAMES)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for i, feat in enumerate(top3_nadir):
    dependence_panel(axes[i], feat, expl_nadir, data_raw, FEATURE_NAMES, COLOR_FEAT_NADIR)
plt.suptitle(f'MLP - SHAP dependence - Nadir Bus 2 (top 3 features, coloured by {COLOR_FEAT_NADIR})')
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'dependence_Nadir_Bus2_MLP.png'),
            dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Notes ---')
print(f"RoCoF Bus 2 colours: chosen per panel by strongest |Pearson r| against the panel x-axis.")
print(f"Nadir Bus 2 colours: forced to {COLOR_FEAT_NADIR} on every panel to expose the AC-OPF "
      f"dispatch coupling between converter dispatch and the size of the SG 3 trip.")

## Cell 6 - Cross-Bus Waterfall (Same Scenario, Different Buses)

In [ ]:
buses_plot = [1, 2, 5, 9]
targets_rocof = [f'RoCoF Bus {b}' for b in buses_plot]
targets_nadir = [f'Nadir Bus {b}' for b in buses_plot]

# RoCoF: same scenario (worst system-wide RoCoF) at 4 buses
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
for i, (bus, tname) in enumerate(zip(buses_plot, targets_rocof)):
    plt.sca(axes[i])
    ex = make_explanation_one(all_shap, tname, i_worst_overall_rocof, X_test_raw, FEATURE_NAMES)
    shap.plots.waterfall(ex, show=False)
    axes[i].set_title(f'Bus {bus} - same scenario')
plt.suptitle('MLP - SHAP Waterfall - Same Worst RoCoF Scenario at Different Buses', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_cross_bus_comparison_RoCoF_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

# Nadir: same scenario at 4 buses
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
for i, (bus, tname) in enumerate(zip(buses_plot, targets_nadir)):
    plt.sca(axes[i])
    ex = make_explanation_one(all_shap, tname, i_worst_overall_nadir, X_test_raw, FEATURE_NAMES)
    shap.plots.waterfall(ex, show=False)
    axes[i].set_title(f'Bus {bus} - same scenario')
plt.suptitle('MLP - SHAP Waterfall - Same Worst Nadir Scenario at Different Buses', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'waterfall_cross_bus_comparison_Nadir_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Analytical commentary ---')
print('The same disturbance scenario produces different feature attributions at different buses - direct evidence for locational explainability.')

## Cell 7 - RoCoF vs Nadir Feature Importance at Bus 2 (and Bus 1, Bus 7)

In [ ]:
mean_abs = {}
for t in TARGET_ORDER:
    if t not in all_shap:
        continue
    mean_abs[t] = np.abs(np.array(all_shap[t]['shap_values'])).mean(axis=0)

buses_compare = [2, 1, 7]
for bus in buses_compare:
    rocof_t = f'RoCoF Bus {bus}'
    nadir_t = f'Nadir Bus {bus}'
    if rocof_t not in mean_abs or nadir_t not in mean_abs:
        continue
    df = pd.DataFrame({'Feature': FEATURE_NAMES, 'RoCoF': mean_abs[rocof_t], 'Nadir': mean_abs[nadir_t]})
    df = df.set_index('Feature')
    df.plot(kind='barh', figsize=(10, 6), color=['#d62728', '#1f77b4'], width=0.75)
    plt.xlabel('Mean |SHAP|')
    plt.title(f'MLP - RoCoF vs Nadir feature importance at Bus {bus}')
    plt.legend(title='Metric')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_out, f'RoCoF_vs_Nadir_Bus{bus}_MLP.png'), dpi=300, bbox_inches='tight')
    plt.show()

print('\n--- Analytical commentary ---')
print('Different bars per feature show how the same bus is driven by different features for RoCoF vs Nadir (initial dynamics vs minimum frequency).')

## Cell 8 - Force Plot (Interactive HTML)

In [ ]:
ev_b2 = all_shap['RoCoF Bus 2']['expected_value']
sv_b2 = np.array(all_shap['RoCoF Bus 2']['shap_values'])
features_one = X_test_raw.iloc[i_worst_rocof_b2]
if isinstance(features_one, pd.Series):
    features_one = features_one.values.reshape(1, -1)
else:
    features_one = np.atleast_2d(features_one)

try:
    fp = shap.force_plot(ev_b2, sv_b2[i_worst_rocof_b2], features_one, feature_names=FEATURE_NAMES, matplotlib=False, show=False)
    shap.save_html(os.path.join(fig_out, 'force_plot_RoCoF_Bus2.html'), fp)
    print('Saved: figures/SHAP_local/force_plot_RoCoF_Bus2.html')
except Exception as e:
    print(f'Force plot (single) error: {e}')

# Multi: 20 worst RoCoF Bus 2 scenarios
idx_20 = idx_worst_rocof_b2
n_20 = min(20, len(sv_b2))
idx_20 = np.argsort(y_rocof_b2)[:n_20]
try:
    fp_multi = shap.force_plot(ev_b2, sv_b2[idx_20], X_test_raw.iloc[idx_20].values, feature_names=FEATURE_NAMES, matplotlib=False, show=False)
    shap.save_html(os.path.join(fig_out, 'force_plot_RoCoF_Bus2_20worst.html'), fp_multi)
    print('Saved: figures/SHAP_local/force_plot_RoCoF_Bus2_20worst.html')
except Exception as e:
    print(f'Force plot (multi) error: {e}')

print('\nOpen the HTML files in a browser for interactive force plots.')

## Cell 9 - Scenario Narrative Summary

In [ ]:
def narrative_one(target_name, iloc_idx, all_shap, X_test_raw, y_test, feature_names):
    ev = all_shap[target_name]['expected_value']
    sv = np.array(all_shap[target_name]['shap_values'])[iloc_idx]
    pred = ev + sv.sum()
    actual = y_test[target_name].iloc[iloc_idx] if hasattr(y_test[target_name], 'iloc') else y_test[target_name].values[iloc_idx]
    order = np.argsort(np.abs(sv))[::-1]
    row = X_test_raw.iloc[iloc_idx]
    parts = [f'In scenario (iloc {iloc_idx}), the {target_name} was predicted as {pred:.4f} (actual: {actual:.4f}). '
             f'The baseline (expected) value is {ev:.4f}. ']
    for r in order[:3]:
        fn = feature_names[r]
        val = row[fn] if fn in row.index else row.iloc[r]
        parts.append(f'The dominant driver #{len(parts)} was {fn} at value {val:.2f}, contributing {sv[r]:.4f} to the deviation. ')
    return ''.join(parts)

print('=== Top 3 worst RoCoF Bus 2 scenarios ===')
for rank, iloc in enumerate(idx_worst_rocof_b2[:3], 1):
    print(f'\n--- Scenario {rank} (iloc {iloc}) ---')
    print(narrative_one('RoCoF Bus 2', iloc, all_shap, X_test_raw, y_test, FEATURE_NAMES))

print('\n=== Top 3 worst Nadir Bus 2 scenarios ===')
for rank, iloc in enumerate(idx_worst_nadir_b2[:3], 1):
    print(f'\n--- Scenario {rank} (iloc {iloc}) ---')
    print(narrative_one('Nadir Bus 2', iloc, all_shap, X_test_raw, y_test, FEATURE_NAMES))

print('\n--- Use these narratives in the discussion chapter. ---')

## Cell 10 - Save Summary Statistics

In [ ]:
summary_rows = []
summary_rows.append({'item': 'worst_rocof_bus2_iloc', 'value': int(i_worst_rocof_b2)})
summary_rows.append({'item': 'worst_rocof_sys_iloc', 'value': int(i_worst_rocof_sys)})
summary_rows.append({'item': 'worst_nadir_bus2_iloc', 'value': int(i_worst_nadir_b2)})
summary_rows.append({'item': 'worst_nadir_sys_iloc', 'value': int(i_worst_nadir_sys)})
summary_rows.append({'item': 'typical_rocof_bus2_iloc', 'value': int(typical_rocof_b2)})
summary_rows.append({'item': 'typical_nadir_bus2_iloc', 'value': int(typical_nadir_b2)})
summary_rows.append({'item': 'worst_overall_rocof_iloc', 'value': int(i_worst_overall_rocof)})
summary_rows.append({'item': 'worst_overall_nadir_iloc', 'value': int(i_worst_overall_nadir)})

for iloc, label in [(i_worst_rocof_b2, 'RoCoF_Bus2_worst'), (i_worst_nadir_b2, 'Nadir_Bus2_worst')]:
    sv_r = np.array(all_shap['RoCoF Bus 2']['shap_values'])[iloc]
    sv_n = np.array(all_shap['Nadir Bus 2']['shap_values'])[iloc]
    for i, fn in enumerate(FEATURE_NAMES):
        summary_rows.append({'item': f'shap_RoCoF_Bus2_{label}_{fn}', 'value': sv_r[i]})
        summary_rows.append({'item': f'shap_Nadir_Bus2_{label}_{fn}', 'value': sv_n[i]})

top_contrib = []
for iloc in list(idx_worst_rocof_b2[:3]) + list(idx_worst_nadir_b2[:3]):
    for tname in ['RoCoF Bus 2', 'Nadir Bus 2']:
        sv = np.array(all_shap[tname]['shap_values'])[iloc]
        order = np.argsort(np.abs(sv))[::-1]
        top_contrib.append({'scenario_iloc': iloc, 'target': tname, 'top1': FEATURE_NAMES[order[0]], 'top2': FEATURE_NAMES[order[1]], 'top3': FEATURE_NAMES[order[2]]})

pd.DataFrame(summary_rows).to_csv(os.path.join(fig_out, 'local_analysis_summary.csv'), index=False)
pd.DataFrame(top_contrib).to_csv(os.path.join(fig_out, 'local_analysis_top_contributors.csv'), index=False)
print('Saved: figures/SHAP_local/local_analysis_summary.csv')
print('Saved: figures/SHAP_local/local_analysis_top_contributors.csv')